# 02 — Hüpoteeside testimine

Kolm hüpoteesi testitakse statistiliselt. Iga hüpoteesi jaoks:
1. Nullhüpotees sõnastatakse selgelt
2. Sobiv statistiline test valitakse ja käivitatakse
3. Tulemus tõlgendatakse

| # | Hüpotees |
|---|----------|
| H1 | Sõiduki vanus mõjutab oluliselt läbimise tõenäosust |
| H2 | Premiumbrandid on usaldusväärsemad kui odavad margid (vanust kontrollides) |
| H3 | Tehnoülevaatuspunkti maht ja mitteminemise määr on seotud |

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.linear_model import LogisticRegression
from pathlib import Path

# ── Year selection ────────────────────────────────────────────────────────────
ANALYSIS_YEARS = list(range(2015, 2025))   # 10 years: 2015–2024

YEAR_URLS = {
    2010: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/9b2d3dbe-e35c-4b5f-baed-6990baa408d0/download-s3',
    2011: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/5139abc3-6823-4121-8d2c-0c82928ac8ac/download-s3',
    2012: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/53f52202-e94e-4cb8-9149-4e311e6f2fdb/download-s3',
    2013: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/3f859ecc-7296-4b9c-ae9b-625b95c90ad9/download-s3',
    2014: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/10103f5a-0b6d-46fc-bd16-99a5c433625e/download-s3',
    2015: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/788a9115-a5da-4f47-bbcc-c1c3b644d3b3/download-s3',
    2016: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/f4f0b51e-8343-4832-9c07-4c04448b8f21/download-s3',
    2017: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/b5a08ea9-0fa8-4cb5-b0bc-7109571d8de4/download-s3',
    2018: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/2d018cd8-f3d7-4242-8514-99633120992a/download-s3',
    2019: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/7ba50767-9844-40bb-8561-af89b634e201/download-s3',
    2020: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/6371e8dc-9906-4555-af9f-f927f2ccf938/download-s3',
    2021: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/f3fe9ef1-897c-45b3-b2dd-908b810aae9c/download-s3',
    2022: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/ba317d52-71b7-473d-bc87-aec0cde38434/download-s3',
    2023: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/1943aed4-8e53-4e70-9946-7fc8ad1f7dfe/download-s3',
    2024: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/af5b081a-3db1-495d-90f3-c334a860938a/download-s3',
    2025: 'https://pilv.transpordiamet.ee/s/Iiee4OAYFq4lT1v/download?path=%2F&files=yv_2025.csv',
}

def build_urls(years):
    urls = [YEAR_URLS[y] for y in sorted(years) if y in YEAR_URLS]
    quoted = ', '.join(f"'{u}'" for u in urls)
    return f'[{quoted}]'

URLS = build_urls(ANALYSIS_YEARS)
CSV_OPTS = "delim=',', header=true, encoding='utf-8'"

print(f'✓ Analüüsitavad aastad: {min(ANALYSIS_YEARS)}–{max(ANALYSIS_YEARS)} ({len(ANALYSIS_YEARS)} aastat)')
print('Esimene päring võib võtta 1–2 minutit.')

---
## H1 — Vanuse mõju läbimise tõenäosusele

**Nullhüpotees H₀:** Alla 10-aastaste ja üle 10-aastaste sõidukite esmakordsete läbivaatuse läbimise määrades ei ole statistiliselt olulist erinevust.

**Testid:**
- Joondiagramm: läbimise määr % vs vanus (0–30 a)
- Hii-ruut test (≤10 vs >10 aastat)
- Logistilise regressiooni kõver

In [ ]:
h1_data = duckdb.sql(f"""
    WITH aged AS (
        SELECT
            CAST(SUBSTR(YV_KUUPAEV, 1, 4) AS INTEGER)
                - TRY_CAST(ESMANE_REG_AASTA AS INTEGER)              AS vanus,
            CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END      AS labis
        FROM read_csv_auto({URLS}, {CSV_OPTS})
        WHERE YLEVAATUSLIIK  = 'KORRALINE'
          AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
          AND ESMANE_REG_AASTA IS NOT NULL
          AND TRY_CAST(ESMANE_REG_AASTA AS INTEGER) BETWEEN 1900 AND 2025
    )
    SELECT
        vanus,
        COUNT(*)                     AS kokku,
        SUM(labis)                   AS labis_arv,
        ROUND(100.0 * AVG(labis), 2) AS labimise_protsent
    FROM aged
    WHERE vanus BETWEEN 0 AND 30
    GROUP BY vanus
    HAVING COUNT(*) >= 100
    ORDER BY vanus
""").df()

print(f'Vanusegruppe: {len(h1_data)}')
h1_data.head()

In [ ]:
# Logistic regression trend curve
X_lr = h1_data['vanus'].values.reshape(-1, 1)
y_lr_bin = (h1_data['labimise_protsent'] > h1_data['labimise_protsent'].mean()).astype(int).values
weights  = h1_data['kokku'].values

lr = LogisticRegression(max_iter=1000)
lr.fit(X_lr, y_lr_bin, sample_weight=weights)

vanus_range = np.linspace(0, 30, 200).reshape(-1, 1)
lr_prob = lr.predict_proba(vanus_range)[:, 1]
# Scale to actual pass % range for visual overlay
lo, hi = h1_data['labimise_protsent'].min(), h1_data['labimise_protsent'].max()
lr_scaled = lr_prob * (hi - lo) + lo

fig_h1 = go.Figure()
fig_h1.add_trace(go.Scatter(
    x=h1_data['vanus'], y=h1_data['labimise_protsent'],
    mode='markers+lines', name='Tegelik läbimise %',
    marker=dict(size=8, color='steelblue'),
    hovertemplate='Vanus: %{x} a<br>Läbimise: %{y:.1f}%<br><extra></extra>'
))
fig_h1.add_trace(go.Scatter(
    x=vanus_range.flatten(), y=lr_scaled,
    mode='lines', name='Logistiline trendjoon',
    line=dict(color='orange', dash='dash', width=2)
))
fig_h1.add_vline(x=10, line_dash='dot', line_color='red',
                 annotation_text='10-aastane piir', annotation_position='top right')
fig_h1.update_layout(
    title='H1 — Läbimise määr sõiduki vanuse lõikes (KORRALINE ülevaatused)',
    xaxis_title='Sõiduki vanus ülevaatuse hetkel (aastat)',
    yaxis_title='Läbimise määr (%)',
    height=450
)
fig_h1.show()

In [ ]:
# Chi-square test: ≤10 vs >10 years
young = h1_data[h1_data['vanus'] <= 10]
old   = h1_data[h1_data['vanus'] >  10]

y_pass  = int(young['labis_arv'].sum())
y_fail  = int(young['kokku'].sum()) - y_pass
o_pass  = int(old['labis_arv'].sum())
o_fail  = int(old['kokku'].sum()) - o_pass

contingency = np.array([[y_pass, y_fail], [o_pass, o_fail]])
chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
n = contingency.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print('=== Hii-ruut test: ≤10 vs >10 aastat ===')
print(f'  ≤10 a:  {y_pass:>8,} läbis | {y_fail:>8,} kukkus | pass% = {100*y_pass/(y_pass+y_fail):.1f}%')
print(f'  >10 a:  {o_pass:>8,} läbis | {o_fail:>8,} kukkus | pass% = {100*o_pass/(o_pass+o_fail):.1f}%')
print(f'\nChi2 = {chi2:.2f},  p = {p_val:.2e},  df = {dof}')
print(f"Cramér's V = {cramers_v:.4f} ({'nõrk' if cramers_v < 0.1 else 'mõõdukas' if cramers_v < 0.3 else 'tugev'})")

if p_val < 0.05:
    print('\n✓ NULLHÜPOTEES LÜKATUD ÜMBER — vanus mõjutab läbimist statistiliselt oluliselt.')
else:
    print('\n✗ Nullhüpoteesi ei saa ümber lükata (p ≥ 0.05).')

### H1 Järeldus

- **Nullhüpotees:** ≤10 a ja >10 a sõidukite läbimise määrdes pole olulist vahet
- **Test:** Hii-ruut kontingentsitabel
- **Tulemus:** *(täitke pärast käivitamist)*
- **Efekti suurus:** Cramér's V = *(väärtus)*
- **Praktiline tähendus:** *(tõlgendage)*

---
## H2 — Margi usaldusväärsus (premium vs eelarvemargid)

**Nullhüpotees H₀:** Premium-margid (BMW, Mercedes-Benz, Audi, Volvo, Lexus jt) ei oma kõrgemat esmakordsete läbimiste määra kui eelarve-margid (Dacia, Chevrolet, Daewoo, Lada jt), kui kontrollida sõiduki vanust (5–15 a sõidukid).

**Testid:**
- Tulpdiagramm: läbimise määr Top 40 marga lõikes
- Vanusega kontrollitud võrdlus (ainult 5–15 a sõidukid)
- Mann-Whitney U test

In [ ]:
PREMIUM_BRANDS = ['BMW', 'MERCEDES-BENZ', 'AUDI', 'VOLVO', 'LEXUS', 'PORSCHE', 'JAGUAR', 'LAND ROVER']
BUDGET_BRANDS  = ['DACIA', 'CHEVROLET', 'DAEWOO', 'LADA', 'SEAT', 'KIA', 'HYUNDAI']

def classify_brand(mark):
    if mark in PREMIUM_BRANDS: return 'Premium'
    if mark in BUDGET_BRANDS:  return 'Eelarve'
    return 'Muu'

h2_all = duckdb.sql(f"""
    WITH aged AS (
        SELECT
            UPPER(MARK)                                                  AS mark,
            CAST(SUBSTR(YV_KUUPAEV, 1, 4) AS INTEGER)
                - TRY_CAST(ESMANE_REG_AASTA AS INTEGER)                 AS vanus,
            CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END         AS labis
        FROM read_csv_auto({URLS}, {CSV_OPTS})
        WHERE YLEVAATUSLIIK  = 'KORRALINE'
          AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
          AND MARK IS NOT NULL AND MARK != ''
          AND ESMANE_REG_AASTA IS NOT NULL
          AND TRY_CAST(ESMANE_REG_AASTA AS INTEGER) BETWEEN 1900 AND 2025
    )
    SELECT
        mark,
        COUNT(*)                            AS kokku,
        SUM(labis)                          AS labis_arv,
        ROUND(100.0 * AVG(labis), 2)        AS labimise_protsent,
        ROUND(AVG(vanus), 1)                AS kesk_vanus
    FROM aged
    WHERE vanus BETWEEN 0 AND 30
    GROUP BY mark
    HAVING COUNT(*) >= 500
    ORDER BY kokku DESC
    LIMIT 40
""").df()

h2_all['segment'] = h2_all['mark'].apply(classify_brand)
color_map = {'Premium': '#2E86AB', 'Eelarve': '#E84855', 'Muu': '#AAAAAA'}

fig_h2 = px.bar(
    h2_all.sort_values('labimise_protsent'),
    x='labimise_protsent', y='mark',
    color='segment', color_discrete_map=color_map,
    orientation='h',
    hover_data=['kokku', 'kesk_vanus'],
    labels=dict(labimise_protsent='Läbimise %', mark='Automärk', segment='Segment'),
    title='H2 — Läbimise määr marga lõikes (Top 40 marka mahult)',
)
fig_h2.update_layout(height=900, xaxis_range=[60, 100])
fig_h2.show()

In [ ]:
# Age-controlled: only 5–15 year old vehicles
h2_ctrl = duckdb.sql(f"""
    WITH aged AS (
        SELECT
            UPPER(MARK)                                                  AS mark,
            CAST(SUBSTR(YV_KUUPAEV, 1, 4) AS INTEGER)
                - TRY_CAST(ESMANE_REG_AASTA AS INTEGER)                 AS vanus,
            CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END         AS labis
        FROM read_csv_auto({URLS}, {CSV_OPTS})
        WHERE YLEVAATUSLIIK  = 'KORRALINE'
          AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
          AND MARK IS NOT NULL AND MARK != ''
          AND ESMANE_REG_AASTA IS NOT NULL
          AND TRY_CAST(ESMANE_REG_AASTA AS INTEGER) BETWEEN 1900 AND 2025
    )
    SELECT
        mark,
        COUNT(*)                            AS kokku,
        SUM(labis)                          AS labis_arv,
        ROUND(100.0 * AVG(labis), 2)        AS labimise_protsent
    FROM aged
    WHERE vanus BETWEEN 5 AND 15
    GROUP BY mark
    HAVING COUNT(*) >= 200
""").df()

h2_ctrl['segment'] = h2_ctrl['mark'].apply(classify_brand)

premium_rates = h2_ctrl[h2_ctrl['segment'] == 'Premium']['labimise_protsent'].values
budget_rates  = h2_ctrl[h2_ctrl['segment'] == 'Eelarve']['labimise_protsent'].values

print('=== Vanusega kontrollitud võrdlus (5–15 a sõidukid) ===')
print(f'Premium: n={len(premium_rates)}, kesk. läbimise % = {premium_rates.mean():.1f}%')
print(f'Eelarve: n={len(budget_rates)},  kesk. läbimise % = {budget_rates.mean():.1f}%')

# Box plot
compare_df = h2_ctrl[h2_ctrl['segment'].isin(['Premium', 'Eelarve'])].copy()
fig_h2b = px.box(
    compare_df, x='segment', y='labimise_protsent',
    color='segment', color_discrete_map=color_map,
    points='all', hover_data=['mark', 'kokku'],
    labels=dict(segment='Segment', labimise_protsent='Läbimise % (5–15 a sõidukid)'),
    title='H2 — Premium vs eelarve margid, vanusega kontrollitud (5–15 a)',
)
fig_h2b.update_layout(height=450, showlegend=False)
fig_h2b.show()

In [ ]:
# Mann-Whitney U test (one-tailed: premium > budget)
if len(premium_rates) >= 3 and len(budget_rates) >= 3:
    u_stat, p_mw = stats.mannwhitneyu(premium_rates, budget_rates, alternative='greater')
    print('=== Mann-Whitney U test (H₀: premium ≤ eelarve) ===')
    print(f'U = {u_stat:.1f},  p = {p_mw:.4f}')
    if p_mw < 0.05:
        print('\n✓ NULLHÜPOTEES LÜKATUD ÜMBER — premium-margid läbivad statistiliselt oluliselt sagedamini.')
    else:
        print('\n✗ Nullhüpoteesi ei saa ümber lükata (p ≥ 0.05).')
else:
    print('Liiga vähe segmente testimiseks.')

### H2 Järeldus

- **Nullhüpotees:** Premium-margid ei oma kõrgemat läbimise määra (vanust kontrollides)
- **Test:** Mann-Whitney U (ühepoolne)
- **Tulemus:** *(täitke pärast käivitamist)*
- **Premium kesk. läbimise %:** *(väärtus)* | **Eelarve kesk. läbimise %:** *(väärtus)*
- **Praktiline tähendus:** *(tõlgendage)*

---
## H3 — Tehnoülevaatuspunkti mahu ja mitteminemise määra seos

**Nullhüpotees H₀:** Tehnoülevaatuspunkti aastane ülevaatuste maht ei ole korreleeritud mitteminemise määraga.

**Testid:**
- Hajuvusdiagramm: maht (x) vs mitteminemise % (y)
- Pearsoni ja Spearmani korrelatsioonikordajad

In [ ]:
h3_data = duckdb.sql(f"""
    SELECT
        TEHNOYLEVAATUSPUNKT                                              AS jaam,
        PUNKTI_KOOD                                                      AS kood,
        COUNT(*)                                                         AS kokku,
        ROUND(100.0 *
            SUM(CASE WHEN YLEVAATUSOTSUS='KORDUVALE' THEN 1 ELSE 0 END)
            / COUNT(*), 2)                                               AS kukkumise_protsent,
        ROUND(100.0 *
            SUM(CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END)
            / COUNT(*), 2)                                               AS labimise_protsent
    FROM read_csv_auto({URLS}, {CSV_OPTS})
    WHERE YLEVAATUSLIIK  = 'KORRALINE'
      AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
      AND TEHNOYLEVAATUSPUNKT IS NOT NULL AND TEHNOYLEVAATUSPUNKT != ''
    GROUP BY TEHNOYLEVAATUSPUNKT, PUNKTI_KOOD
    HAVING COUNT(*) >= 200
    ORDER BY kokku DESC
""").df()

print(f'Punkte (min 200 ülevaatust): {len(h3_data)}')
h3_data.head()

In [ ]:
# Scatter plot with trend line
x = h3_data['kokku'].values
y = h3_data['kukkumise_protsent'].values
trend = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)

fig_h3 = px.scatter(
    h3_data,
    x='kokku', y='kukkumise_protsent',
    size='kokku', color='kukkumise_protsent',
    color_continuous_scale='RdYlGn_r',
    hover_name='jaam',
    hover_data={'kood': True, 'kokku': ':,', 'labimise_protsent': ':.1f', 'kukkumise_protsent': ':.1f'},
    labels=dict(kokku='Ülevaatuste arv kokku', kukkumise_protsent='Mitteminemise %'),
    title='H3 — Ülevaatuspunkti maht vs mitteminemise määr',
)
fig_h3.add_trace(go.Scatter(
    x=x_line, y=np.polyval(trend, x_line),
    mode='lines', name='Trendjoon',
    line=dict(color='black', dash='dash', width=1.5)
))
fig_h3.update_layout(height=550)
fig_h3.show()

In [ ]:
# Correlation tests
r_p, p_p = stats.pearsonr(h3_data['kokku'], h3_data['kukkumise_protsent'])
r_s, p_s = stats.spearmanr(h3_data['kokku'], h3_data['kukkumise_protsent'])

print('=== Korrelatsioon: maht vs mitteminemise % ===')
print(f'Pearsoni  r = {r_p:+.4f},  p = {p_p:.4f}')
print(f'Spearmani r = {r_s:+.4f},  p = {p_s:.4f}')

for name, r, p in [('Pearsoni', r_p, p_p), ('Spearmani', r_s, p_s)]:
    direction = 'positiivne' if r > 0 else 'negatiivne'
    strength  = 'nõrk' if abs(r) < 0.2 else 'mõõdukas' if abs(r) < 0.5 else 'tugev'
    sig = 'statistiliselt oluline' if p < 0.05 else 'EI ole oluline'
    print(f'  {name}: {direction}, {strength} ({sig})')

if p_s < 0.05:
    print('\n✓ NULLHÜPOTEES LÜKATUD ÜMBER — maht ja rangus on statistiliselt oluliselt seotud.')
else:
    print('\n✗ Nullhüpoteesi ei saa ümber lükata (p ≥ 0.05).')

# Fail rate distribution
fig_h3b = px.histogram(
    h3_data, x='kukkumise_protsent', nbins=30,
    color_discrete_sequence=['steelblue'],
    labels=dict(kukkumise_protsent='Mitteminemise %'),
    title='H3 — Mitteminemise % jaotus punktide lõikes',
)
fig_h3b.add_vline(
    x=h3_data['kukkumise_protsent'].mean(), line_dash='dash', line_color='red',
    annotation_text=f"Kesk. {h3_data['kukkumise_protsent'].mean():.1f}%"
)
fig_h3b.update_layout(height=380)
fig_h3b.show()

print(h3_data['kukkumise_protsent'].describe().round(2))

### H3 Järeldus

- **Nullhüpotees:** Punkti maht ei ole korreleeritud mitteminemise määraga
- **Testid:** Pearsoni r ja Spearmani r
- **Tulemus:** *(täitke pärast käivitamist)*
- **Pearsoni r =** *(väärtus)* | **Spearmani r =** *(väärtus)*
- **Praktiline tähendus:** *(näiteks: suuremad punktid rangemad/leebemad?)*

---
## Kokkuvõte

| Hüpotees | Nullhüpotees | Test | Tulemus | p-väärtus |
|----------|-------------|------|---------|----------|
| H1 | Vanus ei mõjuta | Hii-ruut | *(täitke)* | *(täitke)* |
| H2 | Premium = eelarve | Mann-Whitney U | *(täitke)* | *(täitke)* |
| H3 | Maht ei korreleeru rangusega | Pearsoni/Spearmani r | *(täitke)* | *(täitke)* |